# Feature Engineering — Contextualización Colombiana

Aplicamos 6 features adicionales con contexto colombiano: estrato simulado, DTF histórica, capacidad de pago, carga financiera, segmento de edad y riesgo de mora acumulado.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import sys
sys.path.insert(0, '..')

from src.data.preprocess import cargar_datos, limpiar_datos, renombrar_columnas
from src.features.build_features import construir_features

df_base = renombrar_columnas(limpiar_datos(cargar_datos('data/raw/cs-training.csv')))
df = construir_features(df_base)
print(f"Registros: {len(df):,}")
print(f"Columnas nuevas: {set(df.columns) - set(df_base.columns)}")

## Estrato simulado vs tasa de mora

In [ ]:
mora_estrato = df.groupby('estrato_simulado')['default'].agg(['mean', 'count']).reset_index()
mora_estrato.columns = ['estrato', 'tasa_mora', 'registros']

fig = px.bar(mora_estrato, x='estrato', y='tasa_mora',
             title='Tasa de Mora por Estrato Socioeconómico Simulado',
             color='tasa_mora', color_continuous_scale='RdYlGn_r',
             labels={'tasa_mora': 'Tasa de mora', 'estrato': 'Estrato'})
fig.update_layout(yaxis_tickformat='.1%')
fig.show()
print(mora_estrato)

## Capacidad de pago

In [ ]:
fig = px.histogram(
    df[df['capacidad_pago'] < df['capacidad_pago'].quantile(0.99)],
    x='capacidad_pago', color=df['default'].astype(str),
    title='Distribución de Capacidad de Pago por Default',
    barmode='overlay', opacity=0.7,
    color_discrete_map={'0': '#2196F3', '1': '#F44336'},
    labels={'capacidad_pago': 'Ingreso disponible después de deudas'}
)
fig.show()

## Correlación de features nuevos con default

In [ ]:
nuevas = ['estrato_simulado', 'capacidad_pago', 'carga_financiera',
          'riesgo_mora_acumulado', 'tasa_dtf_vigente']
corr_nuevas = df[nuevas + ['default']].corr()['default'].drop('default').sort_values()
print("Correlación de features colombianos con default:")
print(corr_nuevas.to_string())

## Guardar dataset con features

In [ ]:
df.to_parquet('data/processed/dataset_features.parquet', index=False)
print(f"Dataset guardado: data/processed/dataset_features.parquet")
print(f"Shape: {df.shape}")
print(df.dtypes.to_string())